In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
df = pd.read_excel("JEE_2025_Cutoffs.xlsx")
df = df[df["Gender"] == "Gender-Neutral"].reset_index(drop=True)
df

,Institute,Academic Program Name,Quota,Seat Type,Gender,Opening Rank,Closing Rank
0,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",AI,OPEN,Gender-Neutral,10063,13957
1,Indian Institute of Technology Bhubaneswar,"Computer Science and Engineering (4 Years, Bac...",AI,OPEN,Gender-Neutral,2344,3785
2,Indian Institute of Technology Bhubaneswar,"Electrical Engineering (4 Years, Bachelor of T...",AI,OPEN,Gender-Neutral,5585,7242
3,Indian Institute of Technology Bhubaneswar,Electronics and Communication Engineering (4 Y...,AI,OPEN,Gender-Neutral,3873,5535
4,Indian Institute of Technology Bhubaneswar,"Engineering Physics (4 Years, Bachelor of Tech...",AI,OPEN,Gender-Neutral,8840,10494
...,...,...,...,...,...,...,...
1330,Shri G. S. Institute of Technology and Science...,Electronics and Instrumentation Engineering (4...,AI,OPEN,Gender-Neutral,22415,39962
1331,Shri G. S. Institute of Technology and Science...,Electronics and Telecommunication Engineering ...,AI,OPEN,Gender-Neutral,21602,35170
1332,Shri G. S. Institute of Technology and Science...,Industrial and Production Engineering (4 Years...,AI,OPEN,Gender-Neutral,54412,58539
1333,Shri G. S. Institute of Technology and Science...,"Information Technology (4 Years, Bachelor of T...",AI,OPEN,Gender-Neutral,22649,26568


In [3]:
df_iit = df[df["Institute"].str.contains("Indian Institute of Technology")]
df_others = df[~df["Institute"].str.contains("Indian Institute of Technology")]

In [4]:
df.columns

Index(['Institute', 'Academic Program Name', 'Quota', 'Seat Type', 'Gender',
       'Opening Rank', 'Closing Rank'],
      dtype='object')

In [23]:
df_plot = df_iit.copy()

# 1. Data Cleaning
df_plot['Opening Rank'] = pd.to_numeric(df_plot['Opening Rank'], errors='coerce')
df_plot['Closing Rank'] = pd.to_numeric(df_plot['Closing Rank'], errors='coerce')

# Drop rows with NaN ranks (removes preparatory 'P' ranks or other text)
df_clean = df_plot.dropna(subset=['Opening Rank', 'Closing Rank']).copy()

# 2. Define the ranks to check
max_closing_rank = int(df_clean['Closing Rank'].max())
if max_closing_rank > 100000:
    max_closing_rank = 100000

# Generating list: [1, 1000, 2000, 3000, ...]
ranks_to_check = [1] + list(range(500, max_closing_rank + 500, 500))

# 3. Calculate both metrics in a single loop
programs_count = []
colleges_count = []

for rank in ranks_to_check:
    # Find all branches the student qualifies for
    available_seats = df_clean[(df_clean['Opening Rank'] <= rank) & (df_clean['Closing Rank'] >= rank)]
    
    # Metric 1: Count total options (programs)
    programs_count.append(len(available_seats))
    
    # Metric 2: Count UNIQUE institutes
    colleges_count.append(available_seats['Institute'].nunique())

# Create a single DataFrame holding both metrics
results_df = pd.DataFrame({
    'Closing JEE Advanced Rank': ranks_to_check,
    'Opted Programs': programs_count,
    'Opted IITs': colleges_count
})

# 4. Plot using Plotly Express
# Passing a list to 'y' plots both columns as separate, color-coded lines
fig = px.line(
    results_df, 
    x='Closing JEE Advanced Rank', 
    y=['Opted Programs', 'Opted IITs'], 
    markers=True,
    title='Programs and Unique IITs Opted by Rank',
    labels={
        'Target Rank': 'Closing JEE Advanced Rank', 
        'value': 'Count',       # This labels the Y-axis
        'variable': 'Metric'    # This labels the legend
    }
)

# Styling
fig.update_layout(
    xaxis=dict(
        tickmode='linear', 
        dtick=5000, 
        rangemode='nonnegative'
    ), 
    yaxis=dict(
        rangemode='nonnegative'
    ),
    template='plotly_white',
    hovermode="x unified"
)

fig.show()

In [7]:
df["Institute"].nunique()

128

In [8]:
import pandas as pd

# --- Assuming your data is already loaded into 'df' ---
# df = pd.read_csv('josaa_cutoffs.csv')

# 1. Data Cleaning
df['Opening Rank'] = pd.to_numeric(df['Opening Rank'], errors='coerce')
df['Closing Rank'] = pd.to_numeric(df['Closing Rank'], errors='coerce')

# Drop rows with NaN ranks to ensure accurate math
df_clean = df.dropna(subset=['Opening Rank', 'Closing Rank']).copy()


# ==========================================
# 2. DEPARTMENT-WISE STATS (PER INSTITUTE)
# ==========================================
# Group by both Institute and Program to get branch-specific stats
dept_stats_df = df_clean.groupby(['Institute', 'Academic Program Name']).agg(
    Total_Seat_Categories=('Quota', 'count'), # How many different quotas/categories exist for this branch
    Min_Opening_Rank=('Opening Rank', 'min'), # Absolute best rank admitted
    Max_Closing_Rank=('Closing Rank', 'max'), # Absolute lowest rank admitted
    Mean_Opening_Rank=('Opening Rank', 'mean'),
    Std_Opening_Rank=('Opening Rank', 'std'),
    Mean_Closing_Rank=('Closing Rank', 'mean'),
    Std_Closing_Rank=('Closing Rank', 'std')
).reset_index()

# Note: 'std' (Standard Deviation) might be NaN if a branch only has 1 seat category.
# You can fill those NaNs with 0 if you prefer:
dept_stats_df.fillna({'Std_Opening_Rank': 0, 'Std_Closing_Rank': 0}, inplace=True)


# ==========================================
# 3. OVERALL INSTITUTE STATS
# ==========================================
# Group by Institute only to see the overall college profile
institute_stats_df = df_clean.groupby('Institute').agg(
    Total_Programs=('Academic Program Name', 'nunique'), # Count of unique branches
    Institute_Opening_Min=('Opening Rank', 'min'),  
    Institute_Closing_Max=('Closing Rank', 'max'),  
    Institute_Opening_Mean=('Opening Rank', 'mean'),
    Institute_Opening_STD=('Opening Rank', 'std'),
    Institute_Closing_Mean=('Closing Rank', 'mean'),
    Institute_Closing_STD=('Closing Rank', 'std')    
).reset_index()

institute_stats_df['Institute_Opening_STD'] = institute_stats_df['Institute_Opening_STD'].fillna(0)
institute_stats_df['Institute_Closing_STD'] = institute_stats_df['Institute_Closing_STD'].fillna(0)

institute_stats_df['Institute_Opening_STD_Ratio'] = institute_stats_df['Institute_Opening_STD']/institute_stats_df['Institute_Opening_Mean']
institute_stats_df['Institute_Closing_STD_Ratio'] = institute_stats_df['Institute_Opening_STD']/institute_stats_df['Institute_Opening_Mean']

# ==========================================
# 4. OPTIONAL: MERGE THEM TOGETHER
# ==========================================
# If you want one giant master dataframe with both college-level and dept-level stats:
master_stats_df = pd.merge(dept_stats_df, institute_stats_df, on='Institute', how='left')

# Preview the results
# print("--- Department-wise Stats ---")
# print(dept_stats_df.head())

print("\n--- Overall Institute Stats ---")
institute_stats_df.round()


--- Overall Institute Stats ---


,Institute,Total_Programs,Institute_Opening_Min,Institute_Closing_Max,Institute_Opening_Mean,Institute_Opening_STD,Institute_Closing_Mean,Institute_Closing_STD,Institute_Opening_STD_Ratio,Institute_Closing_STD_Ratio
0,"Assam University, Silchar",3,29871,141227,54466.0,26756.0,80060.0,35136.0,0.0,0.0
1,Atal Bihari Vajpayee Indian Institute of Infor...,5,5514,15482,8637.0,2027.0,12127.0,3062.0,0.0,0.0
2,"Birla Institute of Technology, Deoghar Off-Campus",4,33283,101643,58023.0,22772.0,74334.0,22097.0,0.0,0.0
3,"Birla Institute of Technology, Mesra, Ranchi",14,506,92572,34161.0,19670.0,48428.0,26534.0,1.0,1.0
4,"Birla Institute of Technology, Patna Off-Campus",7,20757,87801,48858.0,17084.0,63803.0,15915.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
123,School of Studies of Engineering and Technolog...,10,36709,85380,52856.0,13438.0,71392.0,8836.0,0.0,0.0
124,Shri G. S. Institute of Technology and Science...,9,9767,58539,32366.0,14630.0,40988.0,12068.0,0.0,0.0
125,"Shri Mata Vaishno Devi University, Katra, Jamm...",9,124,77606,42925.0,17719.0,57808.0,22436.0,0.0,0.0
126,University of Hyderabad,2,16128,59393,22828.0,9475.0,44436.0,21152.0,0.0,0.0


In [9]:
institute_stats_df.columns

Index(['Institute', 'Total_Programs', 'Institute_Opening_Min',
       'Institute_Closing_Max', 'Institute_Opening_Mean',
       'Institute_Opening_STD', 'Institute_Closing_Mean',
       'Institute_Closing_STD', 'Institute_Opening_STD_Ratio',
       'Institute_Closing_STD_Ratio'],
      dtype='object')

In [10]:
import plotly.express as px

fig = px.scatter(
    institute_stats_df,
    x='Institute_Closing_Mean',
    y='Total_Programs',
    size='Institute_Closing_STD_Ratio',
    hover_name='Institute',
    title='Institute Landscape: Selectivity vs. Program Variety',
    labels={
        'Institute_Closing_Mean': 'Average Closing Rank (Lower = More Selective)',
        'Total_Programs': 'Number of Programs Offered',
        'Institute_Closing_STD_Ratio': 'Branch Disparity (Rank Spread)'
    }
)
fig.update_layout(template='plotly_white', xaxis=dict(rangemode='nonnegative'), yaxis=dict(rangemode='nonnegative'))
fig.show()

In [11]:
import plotly.express as px

fig = px.scatter(
    institute_stats_df,
    x='Institute_Opening_Mean',
    y='Total_Programs',
    size='Institute_Opening_STD_Ratio',
    hover_name='Institute',
    title='Institute Landscape: Selectivity vs. Program Variety',
    labels={
        'Institute_Opening_Mean': 'Average Opening Rank (Lower = More Selective)',
        'Total_Programs': 'Number of Programs Offered',
        'Institute_Opening_STD_Ratio': 'Branch Disparity (Rank Spread)'
    }
)
fig.update_layout(template='plotly_white', xaxis=dict(rangemode='nonnegative'), yaxis=dict(rangemode='nonnegative'))
fig.show()

In [12]:
# 2. Build the Scatter Plot
fig = px.scatter(
    institute_stats_df,
    x='Institute_Opening_Mean',
    y='Institute_Closing_Mean',
    size='Institute_Opening_STD_Ratio',
    color='Institute_Closing_STD_Ratio',
    hover_name='Institute',
    title='Institute Intake Profile: Opening vs. Closing Rank Dynamics',
    labels={
        'Institute_Opening_Mean': 'Average Opening Rank',
        'Institute_Closing_Mean': 'Average Closing Rank',
        'Institute_Opening_STD_Ratio': 'Opening Rank Spread (Size)',
        'Institute_Closing_STD_Ratio': 'Closing Rank Spread (Color)'
    },
    color_continuous_scale='Viridis' # A clear, modern color gradient
)

# 3. Ensure blobs with 0 or small STD don't become completely invisible
fig.update_traces(marker=dict(sizemin=6))

# 4. Enforce axes to start at 0 and disable negative ranges
fig.update_layout(
    template='plotly_white',
    xaxis=dict(rangemode='nonnegative'),
    yaxis=dict(rangemode='nonnegative'),
    hovermode='closest'
)

fig.show()

In [13]:
rank = 12000
df_rank = df[(df["Opening Rank"] < 1.1*rank) & (df["Closing Rank"] > 0.9*rank)]
df_rank.sort_values(by="Closing Rank")

,Institute,Academic Program Name,Quota,Seat Type,Gender,Opening Rank,Closing Rank
885,Sardar Vallabhbhai National Institute of Techn...,"Artificial Intelligence (5 Years, Integrated B...",OS,OPEN,Gender-Neutral,7777,10823
381,Motilal Nehru National Institute of Technology...,"Electrical Engineering (4 Years, Bachelor of T...",HS,OPEN,Gender-Neutral,9292,10846
507,National Institute of Technology Hamirpur,"Computer Science and Engineering (4 Years, Bac...",OS,OPEN,Gender-Neutral,6425,10852
357,Maulana Azad National Institute of Technology ...,"Computer Science and Engineering (4 Years, Bac...",HS,OPEN,Gender-Neutral,5601,10863
696,"National Institute of Technology, Kurukshetra","Information Technology (4 Years, Bachelor of T...",HS,OPEN,Gender-Neutral,8245,10866
...,...,...,...,...,...,...,...
1221,International Institute of Information Technol...,Electronics and Telecommunication Engineering ...,AI,OPEN,Gender-Neutral,5244,38133
1220,International Institute of Information Technol...,Electrical and Electronics Engineering (4 Year...,AI,OPEN,Gender-Neutral,9820,38848
613,National Institute of Technology Raipur,"Bio Technology (4 Years, Bachelor of Technology)",OS,OPEN,Gender-Neutral,11421,43061
1322,"Rajiv Gandhi National Aviation University, Fur...","Aerospace Engineering (4 Years, Bachelor of Te...",AI,OPEN,Gender-Neutral,8837,50676
